In [ ]:
# play around with ESMFold

In [1]:
import torch
from transformers import AutoTokenizer, EsmForProteinFolding

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1").to(device)
tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model.eval()

Using device: cpu


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

EsmForProteinFolding(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 2560, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 2560, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-35): 36 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=2560, out_features=2560, bias=True)
              (key): Linear(in_features=2560, out_features=2560, bias=True)
              (value): Linear(in_features=2560, out_features=2560, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=2560, out_features=2560, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((2560,), eps=1

In [5]:
# load kpc-11 seq
import pandas as pd
import numpy as np

df = pd.read_csv("kpc11.csv")
seq = df["Sequence"].iloc[0]
seq

'MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGVYAMDTGSGATVSYRAEERFPLCSSFKGFLAAAVLARSQQQAGLLDTPIRYGKNALVLWSPISEKYLTTGMTVAELSAAAVQYSDNAAANLLLKELGGPAGLTAFMRSIGDTTFRLDRWELELNSAIPGDARDTSSPRAVTESLQKLTLGSALAAPQRQQFVDWLKGNTTGNHRIRAAVPADWAVGDKTGTCGVYGTANDYAVVWPTGRAPIVLAVYTRAPNKDDKHSEAVIAAAARLALEGLGVNGQ'

In [7]:
# predict 3D structure
inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=False)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [8]:
with torch.no_grad():
    output = model(**inputs)

In [9]:
# compute pLDDT confidence scores
plddt_mean = output.plddt.mean().item()
print(f"Mean pLDDT confidence score: {plddt_mean:.2f}")

Mean pLDDT confidence score: 0.86


plDDT predicts model confidence estimate for the structure.  
90-100 very high confidence   
70-90 confident   
50-70 low confidence   
0-50 very low confidence.  

In [10]:
#convert to pdb file
pds = model.output_to_pdb(output)
with open("kpc11_predicted.pdb", "w") as f:
    f.write(pds[0])


In [11]:
# full funciton
def predict_structure(seq: str, output_file: str) -> str:
    inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model(**inputs)

    pdbs = model.output_to_pdb(output)
    with open(output_file, "w") as f:
        f.write(pds[0])
    print(f"Predicted structure saved to {output_file}, pLDDT: {output.plddt.mean().item():.2f}")
    return pdbs[0]

In [14]:
mutations = pd.read_csv("mutant_kpc11_probabilities.csv")
mutations.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit,Mean_Delta_Logit,EmbeedingType,Delta_Probability,Mean_Delta_Probability,Min_Logit,Max_Logit
0,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,E,"[[0.05796727538108826, -0.06942269206047058, -...","[[0.08032112568616867, -0.03459315374493599, 0...",1280,0.255149,-1.071336,Non-carbapenemase,-2.876197,0.341683,-0.739574,-2.544435,Mean,-0.603590,-0.517057,-1.886602,1.37362
1,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,L,"[[0.06107009947299957, -0.06913203001022339, -...","[[0.07972662150859833, -0.034166812896728516, ...",1280,0.131632,-1.886602,Non-carbapenemase,-3.691463,0.341683,-0.739574,-2.544435,Mean,-0.727107,-0.517057,-1.886602,1.37362
2,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,K,"[[0.05409065634012222, -0.06780257821083069, -...","[[0.08035734295845032, -0.03513117879629135, 0...",1280,0.797964,1.373620,Carbapenemase,-0.431241,0.341683,-0.739574,-2.544435,Mean,-0.060775,-0.517057,-1.886602,1.37362
3,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,Q,"[[0.057415228337049484, -0.06862905621528625, ...","[[0.0809495598077774, -0.03441504016518593, 0....",1280,0.320343,-0.752197,Non-carbapenemase,-2.557058,0.341683,-0.739574,-2.544435,Mean,-0.538397,-0.517057,-1.886602,1.37362
4,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,V,"[[0.059363480657339096, -0.0705282986164093, -...","[[0.07907991856336594, -0.03507959470152855, 0...",1280,0.163617,-1.631557,Non-carbapenemase,-3.436419,0.341683,-0.739574,-2.544435,Mean,-0.695122,-0.517057,-1.886602,1.37362


In [21]:
# combine pdb file with mean delta logit for KPC-11
from Bio.PDB import PDBParser, PDBIO
import pandas as pd

mean_scores = (
    mutations.groupby("Mutated_position")["Delta_Logit"]
      .mean()
      .to_dict()
)

min_scores = (
    mutations.groupby("Mutated_position")["Delta_Logit"]
      .min()
      .to_dict()
)

parser = PDBParser(QUIET=True)
structure = parser.get_structure("kpc11", "kpc11_predicted.pdb")

for model in structure:
    for chain in model:
        for residue in chain:
            resnum = residue.id[1]
            score = mean_scores.get(resnum, 0.0)
            for atom in residue:
                atom.set_bfactor(float(score))

io = PDBIO()
io.set_structure(structure)
io.save("kpc11_esmfold_mean_delta_logit_bfactor.pdb")

In [22]:
import matplotlib.cm as cm
import matplotlib.colors as colors

norm = colors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=1)
cmap = cm.get_cmap("PuOr_r")

/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_58647/4258623932.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("PuOr_r")


In [118]:
import py3Dmol

def show_structure(score_dict, title):

    with open("kpc11_predicted.pdb") as f:
        pdb_str = f.read()

    view = py3Dmol.view(width=2000, height=2000)
    view.addModel(pdb_str, "pdb")

    # flat lighting
    view.setViewStyle({
        "ambientOcclusion": False,
        "outline": False
    })

    # base cartoon
    view.setStyle({
        "cartoon": {
            "color": "lightgrey",
            "specular": 0,
            "shininess": 0,
            "ambient": 1.0
        }
    })

    # color backbone by score
    for resi, score in score_dict.items():

        rgba = cmap(norm(score))
        r, g, b, _ = rgba
        color = f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"

        view.addStyle(
            {"resi": str(resi)},
            {"cartoon": {
                "color": color,
                "specular": 0,
                "shininess": 0,
                "ambient": 1.0
            }}
        )

    # key residues
    top_positions = [70,114,249,255,256,257,261,271]

    for pos in top_positions:
        view.addStyle(
            {"resi": str(pos)},
            {
                "stick": {"color": "red", "radius": 0.35},
                "sphere": {"color": "red", "radius": 0.5}
            }
        )

    # generate labels from dataframe
    labels = {}

    for pos in top_positions:
        aa = mutations.loc[
            mutations["Mutated_position"] == pos,
            "Canonical_residue"
        ].iloc[0]

        labels[pos] = f"{aa}{pos}"

    # add labels
    for pos, label in labels.items():
        view.addLabel(
            label,
            {
                "fontSize": 20,
                "fontColor": "white",
                "backgroundOpacity": 0.4,
                "borderThickness": 0,
                "inFront": True,
                "alignment": "center",
                "screenOffset": {"x": 4, "y": 4}
            },
            {"resi": str(pos)}
        )

    view.rotate(90,"y")
    view.rotate(20,"x")
    view.zoomTo()

    view.setBackgroundColor("white")

    print(title)

    return view

In [119]:
view = show_structure(
    mean_scores,
    "KPC-11 colored by mean Δlogit"
)

view

KPC-11 colored by mean Δlogit


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [122]:
from IPython.display import Javascript

Javascript("""
var canvas = document.querySelector('canvas');
var link = document.createElement('a');
link.download = 'kpc11_mean_delta_logit_structure.png';
link.href = canvas.toDataURL();
link.click();
""")

<IPython.core.display.Javascript object>

In [120]:
view.png()

In [ ]:
show_structure(min_scores, "KPC-11 colored by min Δlogit", "kpc11_min_delta_logit_structure.png").show()

KPC-11 colored by min Δlogit


3Dmol.js failed to load for some reason. Please check your browser console for error messages.